# 📓 Semana 8 · Dia 3 — CDC com Change Data Feed (CDF)

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEP (CDC) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | CDF habilitado + leitura de mudanças |

---


## 📖 Teoria — O que é CDC e CDF

**CDC (Change Data Capture)**: capturar mudanças nos dados (insert/update/delete) para propagar a outras camadas/sistemas.

O **Change Data Feed (CDF)** do Delta registra **cada mudança** com metadados de operação e versão:

| Coluna | Significado |
|---|---|
| `_change_type` | insert / update_preimage / update_postimage / delete |
| `_commit_version` | versão do commit |
| `_commit_timestamp` | quando |

Uso: alimentar o Ouro incrementalmente, espelhar para outro sistema, auditoria, SCD.


### 💻 Na prática — Habilitando o CDF

Habilite na criação da tabela (recomendado) ou por ALTER.


In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.prata.dim_cliente_cdf (
  CustomerID STRING, nome STRING, cidade STRING)
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true);
INSERT INTO workspace.prata.dim_cliente_cdf VALUES
  ('12345', 'Ana', 'SP'),
  ('67890', 'João', 'RJ');
SHOW TBLPROPERTIES workspace.prata.dim_cliente_cdf;

In [ ]:
# Mudanças: update + delete
spark.sql("UPDATE workspace.prata.dim_cliente_cdf SET cidade = 'CAMPINAS' WHERE CustomerID = '12345'")
spark.sql("DELETE FROM workspace.prata.dim_cliente_cdf WHERE CustomerID = '67890'")
print("Update + delete executados.")

### 💻 Na prática — Lendo o CDF

Leia as mudanças geradas (versão mínima e/ou por timestamp).


In [ ]:
# Ler TODAS as mudanças
mudancas = spark.read.format("delta")\
    .option("readChangeFeed", "true")\
    .table("workspace.prata.dim_cliente_cdf")
mudancas.orderBy("_commit_version").show(truncate=False)

In [ ]:
# Ler mudanças a partir da versão 0 (incremental)
spark.read.format("delta")\
    .option("readChangeFeed", "true")\
    .option("startingVersion", "0")\
    .table("workspace.prata.dim_cliente_cdf")
    .show(truncate=False)

### 💻 Na prática — Consumindo em streaming

Streaming consome o CDF como se fosse uma fonte contínua.


In [ ]:
# Streaming de mudanças (conceito — roda com trigger once no estudo)
stream_mudancas = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .table("workspace.prata.dim_cliente_cdf")
    .writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/bronze/vol_checkpoints/ckpt_cdf")
    .outputMode("append")
    .trigger(once=True)
    .table("workspace.prata.dim_cliente_cdf_historico"))
print("Streaming de CDF configurado (rode e veja as mudanças propagadas).")

> 🎯 **Dica de prova**: DEP: CDF = `readChangeFeed=true` + colunas `_change_type`, `_commit_version`, `_commit_timestamp`. Pergunta: 'como propagar mudanças do Delta para outro sistema?' → CDF.


## 🎯 Exercícios de fixação

**1.** O que `_change_type = update_postimage` significa?

**2.** Habilite CDF em uma tabela existente via ALTER.

**3.** Para que `startingVersion` serve no readChangeFeed?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** update_postimage

A linha APÓS o update — o estado novo. A versão anterior é update_preimage.

**2.** ALTER

```sql
ALTER TABLE t SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
```

**3.** startingVersion

Define a partir de qual versão ler as mudanças — o checkpoint do streaming usa isso para continuar de onde parou.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*